# Running the Experiment

This experiment is implemented as a self-contained training pipeline that can be executed end-to-end with minimal manual intervention. It is designed to fine-tune a CodeBERT-based sequence classification model on the SemEval-2026 Task 13 dataset under a cross-language evaluation setup, while also supporting checkpoint recovery and result tracking within the current run directory.

When the code is executed, the following steps are performed:

1. **Library import and environment setup.**  
   The script begins by importing all required libraries, including standard Python utilities, NumPy, PyTorch, Hugging Face `datasets`, `transformers`, and the `f1_score` metric from scikit-learn.

2. **Device selection and seed initialization.**  
   The computation device is automatically selected. If a CUDA-compatible GPU is available, the experiment runs on GPU; otherwise, it falls back to CPU. A fixed random seed is then applied to Python, NumPy, and PyTorch in order to improve reproducibility across runs.

3. **Pretrained tokenizer setup.**  
   The tokenizer associated with `microsoft/codebert-base` is loaded at the beginning of the experiment. This tokenizer is later used to convert raw code snippets into tokenized inputs suitable for the model.

4. **Dataset loading.**  
   The SemEval-2026 Task 13 dataset is loaded from Hugging Face using the dataset name `DaniilOr/SemEval-2026-Task13` and configuration `A`. The script extracts both the training split and the validation split.

5. **Inspection of available programming languages.**  
   The script prints the list of programming languages present in the training data. This step helps verify the multilingual composition of the dataset before training begins.

6. **Definition of cross-language rotations.**  
   Three language-rotation settings are defined. In each case, the model is trained on one or more source programming languages and validated on a different held-out language. This creates a cross-language evaluation framework:  
   - training on Java and validating on C++  
   - training on C++ and validating on Java  
   - training on Java and C++ and validating on Python

7. **Training configuration definition.**  
   The hyperparameters used in this experiment correspond to the best-performing configuration identified in the previous experiment.

8. **Tokenization and preprocessing.**  
   Each example is tokenized using the CodeBERT tokenizer with truncation and fixed-length padding up to 256 tokens. The original target label is copied into the `labels` field expected by Hugging Face sequence classification models.

9. **Language-based dataset filtering.**  
   Before each run, the training and validation datasets are filtered according to the selected rotation. The training split keeps only examples written in the chosen source language(s), while the validation split keeps only examples written in the target held-out language.

10. **Conversion to PyTorch format and DataLoader construction.**  
    After preprocessing, the datasets are converted to PyTorch tensor format and wrapped in `DataLoader` objects. The training loader uses shuffling, while the validation loader remains ordered and deterministic.

11. **Model initialization for each rotation.**  
    For each language rotation, a new `AutoModelForSequenceClassification` instance is initialized from `microsoft/codebert-base` with `num_labels=2`. The model is then moved to the selected device.

12. **Initial freezing of the encoder backbone.**  
    At the beginning of the setup, the entire CodeBERT encoder (model.roberta) is temporarily frozen. Immediately afterward, the top encoder layers specified by the configuration are unfrozen, so training starts with only the last layers of the encoder being trainable.

13. **Selective unfreezing of top encoder layers.**  
    After freezing the encoder, only the last encoder layers specified by the configuration are unfrozen.

14. **Gradual layer unfreezing across epochs.**  
    At the beginning of each epoch, additional encoder layers are progressively unfrozen according to the configured schedule. However, the optimizer is initialized before this process and is not reconfigured afterward.

15. **Separation of encoder and classifier parameters.**  
    Trainable parameters at initialization time are divided into two groups: encoder parameters and classifier parameters. This separation allows assigning different learning rates to the pretrained backbone and the classification head.

16. **Optimizer initialization.**  
    The `AdamW` optimizer is created with two parameter groups. Encoder parameters use a smaller learning rate, while classifier parameters use a larger learning rate in order to adapt the task-specific head more aggressively.

17. **Learning-rate scheduler with warmup.**  
    The total number of training steps is computed from the number of batches and epochs. A linear learning-rate scheduler with warmup is then created using `get_linear_schedule_with_warmup`, so that training starts more smoothly and the learning rate decreases gradually afterward.

18. **Mixed precision training setup.**  
    If training runs on GPU, automatic mixed precision is enabled through `torch.cuda.amp.autocast`, together with `GradScaler`. This improves computational efficiency and reduces GPU memory usage.

19. **Validation metric definition.**  
    Macro F1 is used as the main evaluation metric. During validation, predictions are generated for the full validation set, converted to discrete labels using `argmax`, and compared against the true labels using `f1_score(..., average="macro")`.

20. **Checkpoint save and load utilities.**  
    Utility functions are defined to save and reload checkpoints. Each checkpoint includes the model state, optimizer state, scheduler state, gradient scaler state, the best validation F1 seen so far, the number of consecutive non-improving epochs, and metadata about the current run.

21. **Experiment result storage.**  
    A JSON results file is used to store the progress of the current experiment run. This file records the result of each language rotation, along with the final mean F1 score across rotations and the identifier of the best run.

22. **Unique run directory creation.**  
    At the beginning of execution, a timestamp-based run directory is created. This ensures that outputs from different executions are stored separately, but also implies that results and checkpoints are not automatically reused across independent runs.

23. **Run identification.**  
    Each rotation is associated with a unique textual identifier containing the rotation index, the training language set, and the validation language. This identifier is used in log messages, checkpoint filenames, and result tracking.

24. **Checkpoint restoration if available.**  
    Before starting training for a rotation, the script checks whether a checkpoint exists at the expected path within the current run directory. If one is found, training resumes from the saved state. In practice, since a new run directory is created for each execution, checkpoint recovery only applies within the same run directory and does not automatically persist across separate executions.

25. **Epoch-level training loop execution.**  
    For every epoch, the model is switched to training mode and iterates through all training batches. For each batch, the script performs forward propagation, loss computation, backward propagation with gradient scaling, gradient clipping, optimizer stepping, scaler updating, and scheduler stepping.

26. **Validation after each epoch.**  
    At the end of every epoch, the model is evaluated on the validation set. The script computes the macro F1 score on validation and the average training loss for that epoch, then prints both values together with the best validation score obtained so far.

27. **Best-score tracking.**  
    After validation, the current F1 score is compared against the best F1 seen so far during that run. If performance improves, the best score is updated and the no-improvement counter is reset.

28. **Early stopping.**  
    If the validation F1 score fails to improve for a predefined number of consecutive epochs, training stops early. This prevents unnecessary training once performance plateaus.

29. **Checkpoint saving after every epoch.**  
    At the end of every epoch, a checkpoint is saved regardless of whether validation performance improved. This makes the experiment fault-tolerant and allows interrupted runs to be resumed later.

30. **Execution across all language rotations.**  
    The full training procedure is repeated for all predefined language-rotation settings. For each rotation, the best validation F1 score is collected and stored.

31. **Per-rotation result persistence.**  
    After each completed rotation, the result is written into the JSON results file together with the training languages, validation language, configuration used, best F1 score, and checkpoint path.

32. **Aggregation of overall performance.**  
    Once all rotations are finished, the script computes the mean F1 score across all rotations. This serves as the overall summary statistic for the experiment.

33. **Best run identification.**  
    Among all completed rotations, the script selects the run with the highest best validation F1 score and stores its identifier in the results summary.

34. **Final best-model reconstruction and export.**  
    If a best run exists, the checkpoint corresponding to that run is loaded, a fresh `AutoModelForSequenceClassification` instance is reconstructed, and the saved weights are loaded into it. The resulting model is then exported as a standalone final file containing the model weights and metadata about the best experiment outcome.

35. **Final reporting.**  
    At the end of execution, the script prints the F1 scores obtained on all rotations, the mean F1 score, the identifier of the best run, the path to the JSON results file, and the path to the final exported best model.

In [ ]:

import os
import json
import time
import random
from typing import Dict, Any, List, Tuple, Union

import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

from datasets import load_dataset
from sklearn.metrics import f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

BASE_MODEL_NAME = "microsoft/codebert-base"
MAX_LEN = 256

DATASET_NAME = "DaniilOr/SemEval-2026-Task13"
DATASET_CONFIG = "A"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

dataset_initial = load_dataset(DATASET_NAME, DATASET_CONFIG)
train_all = dataset_initial["train"]
valid_all = dataset_initial["validation"]

print("Limbaje disponibile:", sorted(set(train_all["language"])))
rotations: List[Tuple[List[str], str]] = [
    (["Java"], "C++"),
    (["C++"], "Java"),
    (["Java", "C++"], "Python"),
]

BEST_CFG: Dict[str, Any] = {
    "batch_size": 32,
    "epochs": 6,
    "lr_encoder": 1e-5,
    "lr_classifier": 1e-4,
    "warmup_ratio": 0.06,
    "unfreeze_initial": 4,
    "unfreeze_step": 2,
    "early_stop": 2,
}
def tokenize_fn(examples: Dict[str, List[Any]]) -> Dict[str, Any]:
    t = tokenizer(
        examples["code"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )
    t["labels"] = examples["label"]
    return t
def filter_by_language(ds, languages: Union[str, List[str]]):
    if isinstance(languages, str):
        languages = [languages]
    return ds.filter(lambda x: x["language"] in languages)
def build_loader(ds, batch_size: int, shuffle: bool) -> DataLoader:
    keep_cols = {"code", "label", "language", "generator"}
    remove_cols = [c for c in ds.column_names if c not in keep_cols]

    proc = ds.map(tokenize_fn, batched=True, remove_columns=remove_cols)
    proc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return DataLoader(proc, batch_size=batch_size, shuffle=shuffle)

def freeze_encoder(model) -> None:
    for p in model.roberta.parameters():
        p.requires_grad = False
def unfreeze_last_layers(model, n_last: int) -> None:
    layers = list(model.roberta.encoder.layer)
    total = len(layers)
    start = max(0, total - n_last)

    for i, layer in enumerate(layers):
        allow = i >= start
        for p in layer.parameters():
            p.requires_grad = allow

@torch.no_grad()
def eval_f1_macro(model, valid_loader: DataLoader) -> float:
    model.eval()
    all_preds, all_true = [], []

    for batch in valid_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attn = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        logits = model(input_ids=input_ids, attention_mask=attn).logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().tolist())
        all_true.extend(labels.cpu().tolist())

    return float(f1_score(all_true, all_preds, average="macro"))

def save_checkpoint(path: str, payload: Dict[str, Any]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(payload, path)


def load_checkpoint(path: str):
    if os.path.exists(path):
        return torch.load(path, map_location=DEVICE)
    return None

def load_results(path: str) -> Dict[str, Any]:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"runs": {}, "best": {"config": None, "mean_f1": -1, "best_run_key": None}}

def save_results(path: str, results: Dict[str, Any]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def run_key(rot_idx: int, val_lang: str) -> str:
    return f"bestcfg_rot{rot_idx}_val{val_lang}"

def train_one_with_checkpoint(
    train_ds,
    valid_ds,
    cfg: Dict[str, Any],
    ckpt_path: str,
    meta_tag: str = "",
) -> float:
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME,
        num_labels=2,
    ).to(DEVICE)

    train_loader = build_loader(train_ds, cfg["batch_size"], shuffle=True)
    valid_loader = build_loader(valid_ds, cfg["batch_size"], shuffle=False)

    freeze_encoder(model)
    unfreeze_last_layers(model, cfg["unfreeze_initial"])

    enc_params, clf_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "classifier" in name:
            clf_params.append(p)
        else:
            enc_params.append(p)

    optimizer = AdamW(
        [
            {"params": enc_params, "lr": cfg["lr_encoder"]},
            {"params": clf_params, "lr": cfg["lr_classifier"]},
        ]
    )

    total_steps = len(train_loader) * cfg["epochs"]
    warmup_steps = int(total_steps * cfg["warmup_ratio"])

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_f1 = 0.0
    no_improve = 0
    start_epoch = 0

    ckpt = load_checkpoint(ckpt_path)
    if ckpt is not None:
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["opt_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        if DEVICE.type == "cuda" and ckpt.get("scaler_state") is not None:
            scaler.load_state_dict(ckpt["scaler_state"])

        best_f1 = float(ckpt.get("best_f1", 0.0))
        no_improve = int(ckpt.get("no_improve", 0))
        start_epoch = int(ckpt.get("epoch", -1)) + 1

        print(f"[{meta_tag}] Checkpoint găsit. Reiau de la epoca {start_epoch + 1}. Best F1={best_f1:.4f}")
    else:
        print(f"[{meta_tag}] Nu există checkpoint. Încep de la epoca 1.")

    for epoch in range(start_epoch, cfg["epochs"]):
        unfreeze_last_layers(model, cfg["unfreeze_initial"] + epoch * cfg["unfreeze_step"])

        model.train()
        losses = []

        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                out = model(
                    input_ids=batch["input_ids"].to(DEVICE),
                    attention_mask=batch["attention_mask"].to(DEVICE),
                    labels=batch["labels"].to(DEVICE),
                )
                loss = out.loss

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            losses.append(float(loss.item()))

        f1 = eval_f1_macro(model, valid_loader)
        mean_loss = float(np.mean(losses)) if losses else 0.0

        print(
            f"[{meta_tag}] Epoca {epoch+1}/{cfg['epochs']} | "
            f"loss={mean_loss:.4f} | F1={f1:.4f} | best={best_f1:.4f}"
        )

        if f1 > best_f1:
            best_f1 = f1
            no_improve = 0
        else:
            no_improve += 1

        payload = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "opt_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict() if DEVICE.type == "cuda" else None,
            "best_f1": best_f1,
            "no_improve": no_improve,
            "meta": {"cfg": cfg, "meta_tag": meta_tag},
        }
        save_checkpoint(ckpt_path, payload)

        if no_improve >= cfg["early_stop"]:
            print(f"[{meta_tag}] Early stop: fără îmbunătățire {no_improve} epoci la rând.")
            break

    return float(best_f1)
run_stamp = time.strftime("%Y%m%d_%H%M%S")
run_dir = f"./runs_bestcfg_{run_stamp}"
results_path = os.path.join(run_dir, "results_bestcfg.json")

results = load_results(results_path)
print("Folder rulare:", run_dir)

scores: List[float] = []

for rot_idx, (train_langs, val_lang) in enumerate(rotations):
    train_ds = filter_by_language(train_all, train_langs)
    valid_ds = filter_by_language(valid_all, val_lang)

    meta = f"bestcfg_rot{rot_idx}_train{'-'.join(train_langs)}_val{val_lang}"
    ckpt_path = os.path.join(run_dir, f"{meta}.pt")

    f1 = train_one_with_checkpoint(
        train_ds=train_ds,
        valid_ds=valid_ds,
        cfg=BEST_CFG,
        ckpt_path=ckpt_path,
        meta_tag=meta,
    )

    scores.append(f1)
    print(f"Train={train_langs} | Val={val_lang} | F1(best)={f1:.4f}")

    k = run_key(rot_idx, val_lang)
    results["runs"][k] = {
        "rot_idx": rot_idx,
        "val_language": val_lang,
        "train_languages": train_langs,
        "config": BEST_CFG,
        "best_f1": float(f1),
        "checkpoint_path": ckpt_path,
    }
    save_results(results_path, results)

mean_f1 = float(np.mean(scores)) if scores else -1.0
best_run_key = max(results["runs"], key=lambda kk: results["runs"][kk]["best_f1"]) if results["runs"] else None

results["best"] = {
    "config": BEST_CFG,
    "mean_f1": mean_f1,
    "best_run_key": best_run_key,
}
save_results(results_path, results)

print("\n==========================")
print("F1 pe rotații:", [float(x) for x in scores])
print("F1 mediu:", mean_f1)
print("Best run key:", best_run_key)
print("Fișier rezultate:", results_path)

if best_run_key is not None:
    best_run = results["runs"][best_run_key]
    best_ckpt_path = best_run["checkpoint_path"]

    ckpt_best = torch.load(best_ckpt_path, map_location=DEVICE)

    best_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME, num_labels=2
    )
    best_model.load_state_dict(ckpt_best["model_state"])
    best_model.to(DEVICE)
    best_model.eval()

    best_model_path = os.path.join(run_dir, f"BEST_MODEL_{run_stamp}.pt")
    torch.save(
        {
            "model_state": best_model.state_dict(),
            "best": results["best"],
            "best_run": best_run,
            "base_model": BASE_MODEL_NAME,
            "num_labels": 2,
        },
        best_model_path,
    )

    print("Salvat model final:", best_model_path)
else:
    print("Nu am putut determina best_run_key (nu există rulari salvate).")


## Expected Output (Key Reproducibility Markers)

During execution, the script prints training logs for each rotation, including loss, Macro F1, best score tracking, and early stopping decisions.

Some non-critical warnings may appear (e.g., missing `HF_TOKEN`, newly initialized classifier weights, deprecated AMP API calls), but these do not affect correctness.

The dataset is loaded successfully and the available languages are:

```text
Limbaje disponibile: ['C++', 'Java', 'Python']

A representative training run produces logs similar to the following:

[bestcfg_rot0_trainJava_valC++] Nu există checkpoint. Încep de la epoca 1.
[bestcfg_rot0_trainJava_valC++] Epoca 1/6 | loss=0.5452 | F1=0.5645 | best=0.0000
[bestcfg_rot0_trainJava_valC++] Epoca 2/6 | loss=0.2730 | F1=0.6888 | best=0.5645
[bestcfg_rot0_trainJava_valC++] Epoca 3/6 | loss=0.2045 | F1=0.6945 | best=0.6888
[bestcfg_rot0_trainJava_valC++] Epoca 4/6 | loss=0.2077 | F1=0.6945 | best=0.6945
[bestcfg_rot0_trainJava_valC++] Epoca 5/6 | loss=0.2074 | F1=0.6945 | best=0.6945
[bestcfg_rot0_trainJava_valC++] Early stop: fără îmbunătățire 2 epoci la rând.
Train=['Java'] | Val=C++ | F1(best)=0.6945

[bestcfg_rot1_trainC++_valJava] Nu există checkpoint. Încep de la epoca 1.
[bestcfg_rot1_trainC++_valJava] Epoca 1/6 | loss=0.5633 | F1=0.6643 | best=0.0000
[bestcfg_rot1_trainC++_valJava] Epoca 2/6 | loss=0.2746 | F1=0.8924 | best=0.6643
[bestcfg_rot1_trainC++_valJava] Epoca 3/6 | loss=0.2273 | F1=0.8361 | best=0.8924
[bestcfg_rot1_trainC++_valJava] Epoca 4/6 | loss=0.2271 | F1=0.8361 | best=0.8924
[bestcfg_rot1_trainC++_valJava] Early stop: fără îmbunătățire 2 epoci la rând.
Train=['C++'] | Val=Java | F1(best)=0.8924

[bestcfg_rot2_trainJava-C++_valPython] Nu există checkpoint. Încep de la epoca 1.
[bestcfg_rot2_trainJava-C++_valPython] Epoca 1/6 | loss=0.4802 | F1=0.3462 | best=0.0000
[bestcfg_rot2_trainJava-C++_valPython] Epoca 2/6 | loss=0.2189 | F1=0.3492 | best=0.3462
[bestcfg_rot2_trainJava-C++_valPython] Epoca 3/6 | loss=0.1939 | F1=0.3469 | best=0.3492
[bestcfg_rot2_trainJava-C++_valPython] Epoca 4/6 | loss=0.1923 | F1=0.3469 | best=0.3492
[bestcfg_rot2_trainJava-C++_valPython] Early stop: fără îmbunătățire 2 epoci la rând.
Train=['Java', 'C++'] | Val=Python | F1(best)=0.3492

F1 pe rotații: [0.6944982046168823, 0.892396518483475, 0.34922820845541575]
F1 mediu: 0.6453743105185911
Best run key: bestcfg_rot1_valJava

Note: Some output messages contain Romanian words.